# Fit PCA

> Fit PCA on pre-encoded embeddings and save reduced representations for generative model training.

In [ ]:
#| default_exp fit_pca

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import os
import torch
import pickle
from pathlib import Path
from tqdm.auto import tqdm
import numpy as np
from sklearn.decomposition import PCA

In [ ]:
#| export
def load_level_embeddings(encoded_dir: Path, split: str, level: int = 0, key: str = 'emb2'):
    """Load all patch embeddings at a given hierarchy level from pre-encoded chunk files.
    Returns (N_total * N_patches, D) — all patches flattened for PCA fitting.
    key: 'emb1', 'emb2', or 'emb3' (which image from the triplet)."""
    chunks = sorted(encoded_dir.glob(f"{split}_chunk*.pt"))
    assert chunks, f"No chunks found for split={split} in {encoded_dir}"
    all_embs = []
    for chunk_path in tqdm(chunks, desc=f"Loading {split} L{level}"):
        chunk = torch.load(chunk_path, weights_only=False)
        for batch_rec in chunk:
            emb = batch_rec[key][level]          # (B, N_patches, D)
            B, P, D = emb.shape
            all_embs.append(emb.reshape(B * P, D))   # flatten patches
    return torch.cat(all_embs, dim=0).float().numpy()  # (N_total * N_patches, D)

In [ ]:
#| export
def fit_and_save_pca(encoded_dir: str, output_dir: str, levels: list = None,
                     n_components: int = 20, key: str = 'emb2'):
    """Fit PCA on per-patch training embeddings for each level, then project and save per-chunk.

    PCA is fit on (N_total * N_patches, D) — all patches from all samples, preserving spatial variance.
    Levels where D <= n_components skip PCA (identity) to avoid fitting on hundreds of millions of rows.
    Output files mirror the input chunk naming:
      <output_dir>/{split}_chunk{idx:05d}_pca{n}.pt
    Each file is a dict with keys 'L0', 'L1', ... each a (N_chunk, N_patches, n_components) tensor.
    One file per input chunk.
    """
    encoded_dir = Path(encoded_dir)
    output_dir  = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    # Detect number of levels from first chunk if not specified
    if levels is None:
        sample = torch.load(sorted(encoded_dir.glob("train_chunk*.pt"))[0], weights_only=False)
        levels = list(range(len(sample[0][key])))

    # --- Step 1: fit one PCA per level on all training patches ---
    pcas = {}
    for level in levels:
        # Peek at embedding dim without loading everything
        sample = torch.load(sorted(encoded_dir.glob("train_chunk*.pt"))[0], weights_only=False)
        D = sample[0][key][level].shape[-1]
        n_comp = min(n_components, D)

        if D <= n_components:
            print(f"L{level}: D={D} <= n_components={n_components}, skipping PCA (saving raw embeddings)")
            pcas[level] = None
            continue

        print(f"Fitting PCA(n={n_comp}) on L{level}...")
        train_emb = load_level_embeddings(encoded_dir, "train", level=level, key=key)
        print(f"  shape: {train_emb.shape}")  # (N_total * N_patches, D)
        pca = PCA(n_components=n_comp, whiten=False)
        pca.fit(train_emb)
        var = pca.explained_variance_ratio_.cumsum()[-1]
        print(f"  variance explained: {var:.1%}")
        pca_path = output_dir / f"pca_L{level}_n{n_components}.pkl"
        with open(pca_path, "wb") as f: pickle.dump(pca, f)
        pcas[level] = pca
        del train_emb

    # --- Step 2: project each chunk, save all levels in one dict file ---
    for split in ["train", "val"]:
        chunks = sorted(encoded_dir.glob(f"{split}_chunk*.pt"))
        print(f"\nProjecting {len(chunks)} {split} chunks...")
        for chunk_path in tqdm(chunks, desc=split):
            data = torch.load(chunk_path, weights_only=False)
            out = {}
            for level in levels:
                # stack all batches: (N_chunk, N_patches, D)
                embs = torch.cat([batch_rec[key][level].float() for batch_rec in data], dim=0)
                N, P, D = embs.shape
                if pcas[level] is None:
                    # No PCA: save raw embeddings as-is
                    out[f"L{level}"] = embs
                else:
                    flat = embs.reshape(N * P, D).numpy()
                    proj = pcas[level].transform(flat)                          # (N*P, n_comp)
                    proj = torch.tensor(proj, dtype=torch.float32).reshape(N, P, -1)  # (N, P, n_comp)
                    out[f"L{level}"] = proj
            save_path = output_dir / f"{chunk_path.stem}_pca{n_components}.pt"
            torch.save(out, save_path)
        print(f"  done → {output_dir}")

    return pcas

## Interactive usage

In [ ]:
#| eval: false
# Example: fit PCA(32) on L0 embeddings
pca = fit_and_save_pca(
    encoded_dir='~/datasets/POP909_encoded',
    output_dir='~/datasets/POP909_pca',
    level=0,
    n_components=32,
)

In [ ]:
#| export
#| eval: false
if __name__ == '__main__':
    import argparse
    parser = argparse.ArgumentParser(description='Fit PCA on pre-encoded embeddings')
    parser.add_argument('encoded_dir', help='Directory containing pre-encoded chunk .pt files')
    parser.add_argument('output_dir', help='Directory to save PCA transform and projected embeddings')
    parser.add_argument('--levels', type=int, nargs='+', default=None, help='Hierarchy levels to process (default: all)')
    parser.add_argument('--n_components', type=int, default=20, help='Number of PCA components (default: 20)')
    parser.add_argument('--key', default='emb2', choices=['emb1','emb2','emb3'],
                        help='Which triplet image embeddings to use (default: emb2)')
    args = parser.parse_args()
    fit_and_save_pca(args.encoded_dir, args.output_dir, args.levels, args.n_components, args.key)

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()